# 08 - Density sweep

**Owner:** Member 1  
**Task:** Recall vs. training density (keep 100/50/25/10% of training interactions) for both models on KuaiRand-Pure and KuaiRec, scored on the biased and the unbiased test, using `data.sparsify`. Validation and both test sets stay fixed. Record users, items, edges, density and cold users at each level. Secondary variant: k-core filtering.  
**Inputs:** src/data/ (sparsify), src/models/, src/training/  
**Outputs:** results/08_density_sweep.json

After running: save the numbers to `results/`, append an entry to `LOG.md`, and write the matching report section.


In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import data
import models
import training
import inference

SEEDS = [0, 1, 2]
RESULTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results"))
os.makedirs(RESULTS_DIR, exist_ok=True)


## Setup
Both models share every setting except the graph (fairness rules 1-4). Finished runs are stored in the JSON file and skipped when the notebook is run again.

**Status: not run.** The settings in `CONFIGS` are placeholders: put the best validation settings from notebooks 03 and 04 there before running. One seed first (compute budget); add seeds to `SEEDS` if time allows.

In [ ]:
SEEDS = [0]
SHARED = dict(dim=64, lr=5e-3, reg=1e-4, batch=2048, epochs=100)
CONFIGS = {"mf": dict(model="mf", **SHARED), "lightgcn": dict(model="lightgcn", layers=3, **SHARED)}
DATASETS = {"kuairand": {}, "kuairec": {}}
LEVELS = [1.0, 0.5, 0.25, 0.1]
CORES = [5, 10]
OUT = os.path.join(RESULTS_DIR, "08_density_sweep.json")

runs = json.load(open(OUT)) if os.path.exists(OUT) else {}
print(len(runs), "finished runs loaded")

In [ ]:
def warm_cold_recall(ds, rec):
    """Mean test recall of users with and without training interactions."""
    user_deg, _ = data.degrees(ds)
    has_test = ~np.isnan(rec)
    warm, cold = has_test & (user_deg > 0), has_test & (user_deg == 0)
    return float(rec[warm].mean()), float(rec[cold].mean()) if cold.any() else None

def run(key, ds, model, seed):
    """Train one model on one variant of a dataset and store the result under `key`."""
    if key in runs:
        return
    res, rec, _, _ = training.fit(CONFIGS[model], ds, seed=seed)
    warm, cold = warm_cold_recall(ds, rec)
    runs[key] = dict(model=model, seed=seed, cfg=res["cfg"], stats=data.stats(ds), test=res["test"],
                     test_unbiased=res.get("test_unbiased"), val_recall=res["val_recall"], best_epoch=res["best_epoch"], train_time=res["train_time"],
                     recall_warm_users=warm, recall_cold_users=cold)
    with open(OUT, "w") as f:
        json.dump(runs, f, indent=1)
    print(f"{key}: recall {res['test']['recall']:.4f}, best epoch {res['best_epoch']}, {res['train_time']:.0f}s", flush=True)

## Random subsampling of training interactions
Keep 100/50/25/10% of the training pairs. Validation, test and the id space stay fixed, so users who lose all training pairs become cold users and are still evaluated.

In [ ]:
for name, kw in DATASETS.items():
    for seed in SEEDS:
        full = data.load(name, seed=seed, **kw)
        for level in LEVELS:
            ds = data.sparsify(full, level, seed=seed)
            for model in CONFIGS:
                run(f"{name}|keep={level}|{model}|seed={seed}", ds, model, seed)

## Secondary variant: k-core filtering
Removes low-degree users and items, so the item set and the test set change. These numbers are not comparable with the subsampling levels above.

In [ ]:
for name, kw in DATASETS.items():
    for seed in SEEDS:
        full = data.load(name, seed=seed, **kw)
        for k in CORES:
            ds = data.k_core(full, k)
            for model in CONFIGS:
                run(f"{name}|core={k}|{model}|seed={seed}", ds, model, seed)

## Results

In [ ]:
rows = []
for key, r in runs.items():
    name, variant, model, _ = key.split("|")
    rows.append(dict(dataset=name, variant=variant, model=model, seed=r["seed"], density=r["stats"]["density"],
                     train=r["stats"]["train"], cold_users=r["stats"]["cold_users"], cold_items=r["stats"]["cold_items"],
                     recall=r["test"]["recall"], recall_unbiased=(r.get("test_unbiased") or {}).get("recall"), ndcg=r["test"]["ndcg"], coverage=r["test"]["coverage"],
                     popularity=r["test"]["popularity"], recall_warm=r["recall_warm_users"],
                     recall_cold=r["recall_cold_users"], best_epoch=r["best_epoch"], time=r["train_time"]))
df = pd.DataFrame(rows)
df = df[df.dataset.isin(DATASETS)]
order = [f"keep={l}" for l in LEVELS] + [f"core={k}" for k in CORES]
df["variant"] = pd.Categorical(df.variant, order, ordered=True)

graph = df.groupby(["dataset", "variant"], observed=True)[["train", "density", "cold_users", "cold_items"]].mean()
graph["density"] = graph["density"].map("{:.3%}".format)
graph

In [ ]:
def mean_std(x):
    return f"{x.mean():.4f} ± {x.std(ddof=0):.4f}"

summary = df.groupby(["dataset", "variant", "model"], observed=True).agg(
    recall=("recall", mean_std), recall_unbiased=("recall_unbiased", "mean"), ndcg=("ndcg", mean_std), coverage=("coverage", mean_std),
    recall_warm=("recall_warm", mean_std), recall_cold=("recall_cold", "mean"),
    best_epoch=("best_epoch", "mean"), time_s=("time", "mean")).round(4)
summary

Gap between the models at each level. A gap smaller than the standard deviations is within noise (fairness rule 5).

In [ ]:
wide = df.pivot_table(index=["dataset", "variant", "seed"], columns="model", values="recall", observed=True)
wide["gain"] = wide["lightgcn"] - wide["mf"]
wide["relative_gain"] = wide["gain"] / wide["mf"]
gap = wide.groupby(["dataset", "variant"], observed=True).agg(
    mf=("mf", "mean"), lightgcn=("lightgcn", "mean"), gain_mean=("gain", "mean"), gain_std=("gain", "std"),
    relative_gain=("relative_gain", "mean"), lightgcn_wins=("gain", lambda g: f"{(g > 0).sum()}/{len(g)}"))
gap.round(4)

In [ ]:
LABELS = {"kuairand": "KuaiRand-Pure", "kuairec": "KuaiRec", "mf": "BPR-MF", "lightgcn": "LightGCN"}
sweep = df[df.variant.astype(str).str.startswith("keep")]
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4.5 * len(DATASETS), 3.4), squeeze=False)
for ax, name in zip(axes[0], DATASETS):
    for model in CONFIGS:
        g = sweep[(sweep.dataset == name) & (sweep.model == model)].groupby("variant", observed=True)
        ax.errorbar(g.density.mean() * 100, g.recall.mean(), yerr=g.recall.std(ddof=0), marker="o", capsize=3, label=LABELS[model])
    ax.set_xscale("log")
    ax.set_xlabel("Training density (%), log scale")
    ax.set_ylabel("Test Recall@20")
    ax.set_title(LABELS[name]); ax.grid(alpha=0.3, which="both"); ax.legend()
fig.suptitle(f"Recall@20 vs. training density ({len(SEEDS)} seeds)", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "08_density_sweep.png"), dpi=200, metadata={"Software": None})
plt.show()

In [ ]:
summary.to_csv(os.path.join(RESULTS_DIR, "08_density_sweep_summary.csv"))
gap.to_csv(os.path.join(RESULTS_DIR, "08_density_sweep_gap.csv"))
print(sorted(os.listdir(RESULTS_DIR)))